# AIC 2026 — OCR keyframe tối ưu cho tiếng Việt (Kaggle 2×T4)

Pipeline ổn định: **CRAFT chỉ phát hiện vùng chữ + VietOCR vgg_transformer nhận dạng**.
Notebook này thay cho bản Paddle-only: recognizer PP-OCRv6 tổng quát trong lần chạy thử chỉ có
3/16 ký tự kiểm tra tiếng Việt, nên có thể trả kết quả sai nhưng confidence vẫn rất cao.

Các bảo vệ chất lượng trong bản này:

- kiểm tra charset VietOCR trước khi cho chạy full;
- padding động theo chiều cao box để không cắt dấu trên/dưới;
- chuẩn hóa Unicode NFC trước khi ghi JSON;
- giữ nguyên acronym như HTV, VTV, THVL và TP.HCM;
- preview rải trên nhiều video và in tỷ lệ ký tự có dấu;
- một process độc lập cho mỗi GPU để tránh hai engine tranh cùng GPU.

> Chạy preview/smoke test trước. Chỉ chạy full khi chữ Việt có dấu đúng bằng mắt.


In [ ]:
!nvidia-smi

# GIỮ NGUYÊN torch/numpy/Pillow/opencv có sẵn của Kaggle. Hai cái bẫy đã gặp trên Colab và
# lặp lại y hệt trên Kaggle:
#   1. `pip install -U easyocr` — cờ -U upgrade cả DEPENDENCY, kéo torch/numpy/Pillow lên bản
#      mới rồi vỡ CUDA. Bỏ -U thì pip thấy dep đã thoả và để nguyên.
#   2. `pip install vietocr` — vietocr 0.3.13 pin `pillow==10.2.0`, hạ cấp Pillow ngay trên cây
#      PIL/ đang dùng và để lại file lẫn version:
#        ImportError: cannot import name 'is_directory' from 'PIL._util'
#      Dùng --no-deps: các dep bị bỏ (pillow, imgaug, albumentations, lmdb, prefetch-generator,
#      scikit-image) chỉ cần khi TRAIN vietocr. Inference chỉ cần torch/numpy/PIL/yaml/einops/gdown.
!pip -q install easyocr
!pip -q install --no-deps vietocr
!python -c "import einops" 2>/dev/null || pip -q install --no-deps einops gdown

## 1. Cấu hình

`TARGET_FOLDERS = None` = chạy **toàn bộ** thư mục `Keyframes*` tìm thấy, đúng như yêu cầu.
Dataset được dò tự động vì tên mount trên Kaggle hay lồng thêm vài cấp.

In [ ]:
import json
import os
import shutil
from pathlib import Path

IS_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or Path('/kaggle/input').exists()
assert IS_KAGGLE, 'Notebook này dành cho Kaggle. Bản Colab: AIC_OCR_Keyframes_EasyOCR_VietOCR_Colab.ipynb'

OUTPUT_ROOT = Path('/kaggle/working/OCR_Vietnamese_CRAFT_VietOCR')
# /kaggle/temp KHÔNG bị đưa vào output khi Save Version -> để weight và ảnh preview ở đây cho
# /kaggle/working sạch, chỉ chứa kết quả OCR.
SCRATCH = Path('/kaggle/temp') if Path('/kaggle/temp').is_dir() else Path('/tmp')
MODEL_DIR = SCRATCH / 'ocr_models'
PREVIEW_DIR = SCRATCH / 'ocr_preview'

# None = mọi thư mục Keyframes* (một session làm hết). Hoặc liệt kê tay để chia đợt.
# Chia 2 đợt theo KHỐI LƯỢNG (không theo tên) vì L26 chiếm ~45% và L25 ~21% dataset —
# cắt kiểu "L21-L25 / L26-L30" sẽ lệch 1:3. Xem mục 10 để biết cách nối tiếp bằng RESUME_FROM.
# --- ĐỢT 1 (~49%) — chạy session này ---
TARGET_FOLDERS = ['Keyframes_L26_a', 'Keyframes_L26_b', 'Keyframes_L26_c',
                  'Keyframes_L26_d', 'Keyframes_L26_e',
                  'Keyframes_L27', 'Keyframes_L23']

# --- ĐỢT 2 (~51%) — session sau, nhớ set RESUME_FROM trỏ tới output đã Save Version của đợt 1 ---
# TARGET_FOLDERS = ['Keyframes_L25', 'Keyframes_L21', 'Keyframes_L22',
#                   'Keyframes_L24', 'Keyframes_L28', 'Keyframes_L29',
#                   'Keyframes_L30']

# Số process = số GPU. Đặt 1 để ép chạy một GPU khi cần so sánh tốc độ.
import torch
NUM_WORKERS = torch.cuda.device_count() or 1

# --- Tham số detection (CRAFT) ---------------------------------------------
# Mặc định của EasyOCR nhắm ảnh chụp; keyframe tin tức thì chữ chạy/chyron NHỎ hơn nhiều.
# mag_ratio 1.5 phóng ảnh trước khi vào CRAFT, low_text 0.3 nới vùng chữ -> bắt được dòng nhỏ.
# Nếu preview cho thấy nhiều box rác, tăng TEXT_THRESHOLD trước, đừng vội hạ mag_ratio.
TEXT_THRESHOLD = 0.6
LOW_TEXT = 0.3
LINK_THRESHOLD = 0.4
CANVAS_SIZE = 2560
MAG_RATIO = 1.5
MIN_SIZE = 10          # CRAFT bỏ box nhỏ hơn ngần này (px)

BOX_PADDING = 4        # padding tối thiểu (px)
BOX_PADDING_X_RATIO = 0.08  # padding ngang theo chiều cao dòng chữ
BOX_PADDING_Y_RATIO = 0.18  # padding dọc lớn hơn để giữ dấu mũ/dấu nặng tiếng Việt
MIN_BOX_SIDE = 12      # crop cao dưới 12px upscale lên 32px chỉ còn nhiễu -> VietOCR đoán bừa

# --- Ngưỡng tin cậy ---------------------------------------------------------
# MIN_CONFIDENCE quyết định box nào vào trường 'text'. KEEP_FLOOR quyết định box nào được GHI
# vào JSON. Giữ khoảng hở giữa hai số: muốn đổi ngưỡng sau này chỉ cần chạy lại mục 6, không
# phải chạy lại GPU.
MIN_CONFIDENCE = 0.35
KEEP_FLOOR = 0.10

# --- Batch / IO (2×T4 16 GB, 4 vCPU) ---------------------------------------
# VietOCR vgg_transformer resize crop về cao 32px nên VRAM gần như không phải giới hạn;
# 64 là mức gom đủ lớn mà không để lô cuối quá lệch. CRAFT ở 2560px mới là chỗ tốn VRAM,
# nhưng ta chạy từng ảnh một nên ~2-3 GB. Còn rất nhiều chỗ trống trong 16 GB.
RECOGNITION_BATCH = 64
IO_WORKERS = 3         # 4 vCPU chia cho 2 process -> 3 thread đọc ảnh mỗi process là vừa
FLUSH_EVERY = 50       # ghi checkpoint mỗi 50 ảnh (đĩa local nên rẻ, khác hẳn Drive)

# Save & Run All bị timeout ở 12h sẽ bị đánh failed và /kaggle/working thường KHÔNG được lưu.
# Dừng sớm ở 10.5h để mục 6-7 kịp chạy. Mốc chỉ kiểm tra GIỮA HAI VIDEO.
MAX_RUN_HOURS = 10.5

# Kết quả của session trước (Save Version -> add output làm input). Ví dụ:
#   RESUME_FROM = ['/kaggle/input/aic-ocr-easyocr-vietocr-p1/OCR_EasyOCR_VietOCR']
RESUME_FROM = []

MODEL_ID = 'craft-det + vietocr-vgg_transformer-vi-v2'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

for d in (OUTPUT_ROOT, MODEL_DIR, PREVIEW_DIR):
    d.mkdir(parents=True, exist_ok=True)


# --- Dò dataset -------------------------------------------------------------
def _looks_like_dataset_root(path):
    """Dataset root = thư mục chứa các Keyframes*/ hoặc map-keyframes*/."""
    if not path.is_dir():
        return False
    try:
        children = list(path.iterdir())
    except (PermissionError, OSError):
        return False
    return any(p.is_dir() and (p.name.startswith('Keyframes') or p.name.startswith('map-keyframes'))
               for p in children)

def _discover_dataset_root():
    seeds = [Path('/kaggle/input/aic-dataset'),
             Path('/kaggle/input/datasets/fatle542/aic-dataset')]
    for seed in seeds:
        if _looks_like_dataset_root(seed):
            return seed
    for pattern in ('*', '*/*', '*/*/*'):
        for candidate in sorted(Path('/kaggle/input').glob(pattern)):
            if _looks_like_dataset_root(candidate):
                return candidate
    return seeds[0]

def _find_map_keyframes(root):
    direct = root / 'map-keyframes-aic25-b1' / 'map-keyframes'
    if direct.is_dir():
        return direct
    for pattern in ('map-keyframes', '*/map-keyframes', 'map-keyframes*/map-keyframes*'):
        for candidate in sorted(root.glob(pattern)):
            if candidate.is_dir():
                return candidate
    return direct

dataset_root = _discover_dataset_root().resolve()
if not dataset_root.is_dir():
    print('KHÔNG thấy dataset. Trong /kaggle/input đang có:')
    for entry in sorted(Path('/kaggle/input').glob('*')):
        print(' ', entry)
        for sub in sorted(entry.glob('*'))[:10]:
            print('    ', sub.name)
assert dataset_root.is_dir(), f'Không thấy dataset root: {dataset_root}'

MAP_KEYFRAMES_DIRECTORY = _find_map_keyframes(dataset_root)

if TARGET_FOLDERS is None:
    KEYFRAME_ROOTS = sorted(p for p in dataset_root.iterdir()
                            if p.is_dir() and p.name.startswith('Keyframes'))
    missing = []
else:
    KEYFRAME_ROOTS, missing = [], []
    for folder in TARGET_FOLDERS:
        selected = (dataset_root / folder.strip()).resolve()
        assert dataset_root in selected.parents, f'Chỉ nhận đường dẫn tương đối: {folder}'
        (KEYFRAME_ROOTS if selected.is_dir() else missing).append(
            selected if selected.is_dir() else folder)

assert KEYFRAME_ROOTS, ('Không thấy thư mục keyframe nào. Trong dataset có: '
                        + ', '.join(sorted(p.name for p in dataset_root.iterdir() if p.is_dir())[:20]))

if missing:
    print('Bỏ qua thư mục không tồn tại:', ', '.join(str(m) for m in missing))
print('Dataset :', dataset_root)
print('Map     :', MAP_KEYFRAMES_DIRECTORY, '' if MAP_KEYFRAMES_DIRECTORY.is_dir() else '(KHÔNG THẤY)')
print('Output  :', OUTPUT_ROOT)
print('GPU     :', NUM_WORKERS, 'x', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Folders :', len(KEYFRAME_ROOTS))

## 2. Liệt kê video + khối lượng việc

`map-keyframes/<video_id>.csv` cho `frame_idx` thật — con số phải nộp cho BTC. OCR phải mang
theo nó chứ không chỉ giữ tên file ảnh.

In [ ]:
import csv
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def find_video_dirs(root):
    base = root / 'keyframes'
    if not base.is_dir():
        base = root
    return [p for p in sorted(base.iterdir()) if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def load_keyframe_map(video_id):
    path = MAP_KEYFRAMES_DIRECTORY / f'{video_id}.csv'
    if not path.is_file():
        return None
    mapping = {}
    with path.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            try:
                mapping[int(row['n'])] = {'pts_time': float(row['pts_time']),
                                          'fps': float(row['fps']),
                                          'frame_idx': int(row['frame_idx'])}
            except (KeyError, TypeError, ValueError):
                pass
    return mapping or None

def keyframe_order(path):
    return int(path.stem) if path.stem.isdigit() else None

def list_images(video_dir):
    return sorted((p for p in video_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
                  key=lambda p: (keyframe_order(p) is None, keyframe_order(p) or 0, p.name))

video_dirs = sorted((v for root in KEYFRAME_ROOTS for v in find_video_dirs(root)), key=lambda p: p.name)
assert video_dirs, 'Không tìm thấy video nào'

IMAGE_COUNTS = {v: len(list_images(v)) for v in video_dirs}
total_images = sum(IMAGE_COUNTS.values())
have_map = sum(load_keyframe_map(v.name) is not None for v in video_dirs)

print(f'{len(video_dirs)} video | {total_images} keyframe | {have_map} video có map-keyframes\n')
print(f'{"thư mục":<22}{"video":>7}{"keyframe":>10}')
for root in KEYFRAME_ROOTS:
    owned = [v for v in video_dirs if root in v.parents]
    print(f'{root.name:<22}{len(owned):>7}{sum(IMAGE_COUNTS[v] for v in owned):>10}')
if have_map < len(video_dirs):
    print(f'\nCẢNH BÁO: {len(video_dirs) - have_map} video thiếu map-keyframes -> frame_idx sẽ là null')

## 3. Nạp sẵn weight **một lần** trong process cha

Hai worker khởi động cùng lúc sẽ cùng tải `craft_mlt_25k.pth` và `vgg_transformer.pth` vào cùng
một thư mục — dễ ghi đè nhau giữa chừng và để lại file hỏng. Tải xong ở đây rồi mới fork thì
worker chỉ đọc cache, không chạm mạng.

In [ ]:
import torch

CRAFT_READY = False
try:
    import easyocr
    # recognizer=False: ta chỉ dùng CRAFT để DETECT, phần đọc chữ là việc của VietOCR.
    # Không tắt thì EasyOCR tải thêm ~100 MB model latin và giữ nó trong VRAM vô ích.
    _warm = easyocr.Reader(['vi'], gpu=False, recognizer=False,
                           model_storage_directory=str(MODEL_DIR), verbose=False)
    del _warm
    CRAFT_READY = True
    print('CRAFT ->', MODEL_DIR)
except Exception as exc:
    print('Tải CRAFT lỗi:', repr(exc))

from vietocr.tool.config import Cfg

VIETOCR_WEIGHTS = MODEL_DIR / 'vgg_transformer.pth'
if not VIETOCR_WEIGHTS.is_file():
    url = Cfg.load_config_from_name('vgg_transformer')['weights']
    print('Tải VietOCR từ', url)
    torch.hub.download_url_to_file(url, str(VIETOCR_WEIGHTS))
size_mb = VIETOCR_WEIGHTS.stat().st_size / 1e6
assert size_mb > 50, f'File weight VietOCR chỉ {size_mb:.1f} MB -> tải hỏng, xoá đi tải lại'
print(f'VietOCR -> {VIETOCR_WEIGHTS} ({size_mb:.0f} MB)')
assert CRAFT_READY, 'Chưa có weight CRAFT — kiểm tra Internet đã bật chưa'


## 4. Kéo kết quả session trước vào (nếu có)

Kaggle giới hạn 12h/session. Muốn chạy tiếp ở session sau: *Save Version* → thêm output của
version đó làm input → điền `RESUME_FROM` ở mục 1. Cell này chép các video đã xong vào
`OUTPUT_ROOT`, worker sẽ tự bỏ qua chúng.

In [ ]:
copied = 0
for source in RESUME_FROM:
    source = Path(source)
    if not source.is_dir():
        print('Bỏ qua (không thấy):', source)
        continue
    for path in sorted(source.glob('L*_V*.json')):
        if path.name.endswith('.partial.json'):
            continue
        target = OUTPUT_ROOT / path.name
        if target.exists():
            continue
        try:
            payload = json.loads(path.read_text(encoding='utf-8'))
        except Exception:
            continue
        # Chỉ nhận video ĐÃ XONG và cùng model — nửa vời thì để worker làm lại từ đầu còn hơn
        # là trộn kết quả của hai cấu hình khác nhau vào một index.
        if payload.get('model') == MODEL_ID and payload.get('complete'):
            shutil.copy2(path, target)
            copied += 1
print(f'Đã chép {copied} video từ {len(RESUME_FROM)} nguồn')

done_ids = {p.stem for p in OUTPUT_ROOT.glob('L*_V*.json') if not p.name.endswith('.partial.json')}
pending = [v for v in video_dirs if v.name not in done_ids]
print(f'\nTổng: {len(video_dirs)} video, đã có {len(done_ids)}, còn phải OCR {len(pending)} '
      f'({sum(IMAGE_COUNTS[v] for v in pending)} keyframe)')

## 5. Worker — chạy trên **một** GPU

Toàn bộ phần OCR nằm trong một file `.py` độc lập thay vì trong notebook, vì `multiprocessing`
kiểu spawn không pickle được hàm định nghĩa trong cell Jupyter. Notebook sẽ khởi động file này
nhiều lần, mỗi lần với `CUDA_VISIBLE_DEVICES` khác nhau.

In [ ]:
%%writefile /kaggle/working/ocr_worker.py
"""OCR một shard video trên MỘT GPU. Notebook cha chạy nhiều bản song song."""
import argparse
import csv
import json
import os
import re
import sys
import time
import unicodedata
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

parser = argparse.ArgumentParser()
parser.add_argument('--config', required=True)
parser.add_argument('--shard-index', type=int, default=0)
parser.add_argument('--shard-count', type=int, default=1)
parser.add_argument('--limit-videos', type=int, default=0)   # >0: smoke test
parser.add_argument('--preview', type=int, default=0)        # >0: chỉ vẽ box rồi thoát
args = parser.parse_args()

CFG = json.loads(Path(args.config).read_text(encoding='utf-8'))
TAG = f'[w{args.shard_index}]'

def log(*parts):
    print(TAG, *parts, flush=True)

OUTPUT_ROOT = Path(CFG['output_root'])
MAP_DIR = Path(CFG['map_dir'])
MODEL_ID = CFG['model_id']
MIN_CONFIDENCE = CFG['min_confidence']
KEEP_FLOOR = CFG['keep_floor']
BOX_PADDING = CFG['box_padding']
BOX_PADDING_X_RATIO = CFG['box_padding_x_ratio']
BOX_PADDING_Y_RATIO = CFG['box_padding_y_ratio']
MIN_BOX_SIDE = CFG['min_box_side']
REC_BATCH = CFG['recognition_batch']
FLUSH_EVERY = CFG['flush_every']
IMAGE_EXTENSIONS = set(CFG['image_extensions'])
DET = CFG['detect']
DEADLINE = time.time() + CFG['max_run_hours'] * 3600 if CFG['max_run_hours'] > 0 else None

# ---------------------------------------------------------------- liệt kê video
VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def find_video_dirs(root):
    base = root / 'keyframes'
    if not base.is_dir():
        base = root
    return [p for p in sorted(base.iterdir()) if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def load_keyframe_map(video_id):
    path = MAP_DIR / f'{video_id}.csv'
    if not path.is_file():
        return None
    mapping = {}
    with path.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            try:
                mapping[int(row['n'])] = {'pts_time': float(row['pts_time']),
                                          'fps': float(row['fps']),
                                          'frame_idx': int(row['frame_idx'])}
            except (KeyError, TypeError, ValueError):
                pass
    return mapping or None

def keyframe_order(path):
    return int(path.stem) if path.stem.isdigit() else None

def list_images(video_dir):
    return sorted((p for p in video_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
                  key=lambda p: (keyframe_order(p) is None, keyframe_order(p) or 0, p.name))

all_videos = sorted((v for root in map(Path, CFG['keyframe_roots']) for v in find_video_dirs(root)),
                    key=lambda p: p.name)
# Round-robin trên danh sách đã sắp xếp -> hai shard cân nhau dù video dài ngắn khác nhau.
my_videos = all_videos[args.shard_index::args.shard_count]

# ---------------------------------------------------------------- model
import numpy as np
import torch
from PIL import Image
import easyocr
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

use_gpu = torch.cuda.is_available()
log('CUDA_VISIBLE_DEVICES=' + str(os.environ.get('CUDA_VISIBLE_DEVICES')),
    '| device:', torch.cuda.get_device_name(0) if use_gpu else 'CPU',
    f'| {len(my_videos)}/{len(all_videos)} video')

detector = easyocr.Reader(['vi'], gpu=use_gpu, recognizer=False,
                          model_storage_directory=CFG['model_dir'],
                          download_enabled=False, verbose=False)

vietocr_config = Cfg.load_config_from_name('vgg_transformer')
vietocr_config['weights'] = CFG['vietocr_weights']
vietocr_config['cnn']['pretrained'] = False       # backbone lấy từ checkpoint, không cần imagenet
vietocr_config['device'] = 'cuda:0' if use_gpu else 'cpu'
vietocr_config['predictor']['beamsearch'] = False # beamsearch chậm gấp nhiều lần, lợi không đáng

# Chặn chạy full nếu checkpoint/config không có đủ các nhóm ký tự tiếng Việt quan trọng.
VIETNAMESE_PROBES = 'ăâêôơưđĂÂÊÔƠƯĐáàảãạấầẩẫậắằẳẵặéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ'
vocab = vietocr_config.get('vocab', '')
missing = [character for character in VIETNAMESE_PROBES if character not in vocab]
assert not missing, 'VietOCR config thiếu charset tiếng Việt: ' + ' '.join(missing)
log(f'VietOCR charset gate: PASS ({len(VIETNAMESE_PROBES)} ký tự kiểm tra)')
recognizer = Predictor(vietocr_config)

# ---------------------------------------------------------------- OCR
def detect_boxes(image):
    horizontal, free = detector.detect(
        np.array(image),
        min_size=DET['min_size'], text_threshold=DET['text_threshold'],
        low_text=DET['low_text'], link_threshold=DET['link_threshold'],
        canvas_size=DET['canvas_size'], mag_ratio=DET['mag_ratio'])
    boxes = []
    for x1, x2, y1, y2 in (horizontal[0] if horizontal else []):
        boxes.append((int(x1), int(y1), int(x2), int(y2)))
    for polygon in (free[0] if free else []):
        xs = [int(p[0]) for p in polygon]
        ys = [int(p[1]) for p in polygon]
        boxes.append((min(xs), min(ys), max(xs), max(ys)))

    width, height = image.size
    cleaned = []
    for x1, y1, x2, y2 in boxes:
        box_height = max(1, y2 - y1)
        pad_x = max(BOX_PADDING, int(round(box_height * BOX_PADDING_X_RATIO)))
        pad_y = max(BOX_PADDING, int(round(box_height * BOX_PADDING_Y_RATIO)))
        x1, y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
        x2, y2 = min(width, x2 + pad_x), min(height, y2 + pad_y)
        if x2 - x1 >= MIN_BOX_SIDE and y2 - y1 >= MIN_BOX_SIDE:
            cleaned.append((x1, y1, x2, y2))
    # Thứ tự đọc: gom theo dải ngang rồi trái->phải. Dải phải co giãn theo chiều cao ảnh,
    # nếu hardcode 20px thì 720p và 1080p cho ra thứ tự khác nhau.
    band = max(12, height // 40)
    return sorted(cleaned, key=lambda b: (b[1] // band, b[0]))


_BATCH_OK = None

def _verify_batch_order(sample):
    """VietOCR gom crop theo bucket chiều rộng trước khi suy luận. Nếu bản đang cài không
    scatter kết quả về đúng index thì text sẽ bị gán nhầm box mà KHÔNG ném exception nào —
    sai âm thầm. Kiểm tra một lần trên ảnh thật rồi quyết định."""
    try:
        batch_texts, _ = recognizer.predict_batch(list(sample), return_prob=True)
        single_texts = [recognizer.predict(crop, return_prob=True)[0] for crop in sample]
    except Exception as exc:
        log('Không kiểm tra được predict_batch:', repr(exc), '-> dùng predict từng ảnh')
        return False
    if list(batch_texts) == single_texts:
        log('predict_batch giữ đúng thứ tự -> bật batch')
        return True
    log('CẢNH BÁO: predict_batch trả SAI thứ tự -> lùi về predict từng ảnh (chậm hơn, nhưng đúng)')
    log('  batch :', list(batch_texts))
    log('  single:', single_texts)
    return False

def recognize_crops(crops):
    global _BATCH_OK
    if not crops:
        return []
    if _BATCH_OK is None and len(crops) >= 3:
        _BATCH_OK = _verify_batch_order(crops[:6])
    if _BATCH_OK is False:
        return [recognizer.predict(crop, return_prob=True) for crop in crops]

    results = []
    for start in range(0, len(crops), REC_BATCH):
        batch = crops[start:start + REC_BATCH]
        try:
            texts, probs = recognizer.predict_batch(batch, return_prob=True)
        except Exception as exc:
            log('predict_batch lỗi, lùi về từng ảnh cho lô này:', repr(exc))
            pairs = [recognizer.predict(crop, return_prob=True) for crop in batch]
            texts, probs = zip(*pairs)
        results.extend(zip(texts, probs))
    return results

def ocr_image(image):
    """Trả về MỌI detection kèm cờ 'kept'. Bộ lọc ngưỡng áp ở tầng trên."""
    boxes = detect_boxes(image)
    detections = []
    for box, (text, confidence) in zip(boxes, recognize_crops([image.crop(b) for b in boxes])):
        text = unicodedata.normalize('NFC', ' '.join((text or '').split())).strip()
        confidence = float(confidence)
        if not text or confidence < KEEP_FLOOR:
            continue
        detections.append({'text': text, 'confidence': round(confidence, 4),
                           'box': list(box), 'kept': confidence >= MIN_CONFIDENCE})
    return detections

# ---------------------------------------------------------------- IO
pool = ThreadPoolExecutor(max_workers=CFG['io_workers'])

def load_image(path):
    try:
        return Image.open(path).convert('RGB')
    except Exception as exc:
        log('Ảnh hỏng:', path, repr(exc))
        return None

def iter_loaded(paths, prefetch=8):
    """Giải mã JPEG chạy trước trên thread khác để GPU không phải đợi đĩa."""
    queue, index = deque(), 0
    while index < len(paths) and len(queue) < prefetch:
        queue.append((paths[index], pool.submit(load_image, paths[index])))
        index += 1
    while queue:
        path, future = queue.popleft()
        if index < len(paths):
            queue.append((paths[index], pool.submit(load_image, paths[index])))
            index += 1
        yield path, future.result()

def atomic_write(path, payload):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
    temp.replace(path)

def output_json_path(video_id):
    return OUTPUT_ROOT / f'{video_id}.json'

def partial_json_path(video_id):
    return OUTPUT_ROOT / f'{video_id}.partial.json'

# ---------------------------------------------------------------- một video
def ocr_video(video_dir):
    video_id = video_dir.name
    mapping = load_keyframe_map(video_id)
    images = list_images(video_dir)

    payload = None
    partial = partial_json_path(video_id)
    if partial.exists():
        try:
            candidate = json.loads(partial.read_text(encoding='utf-8'))
            if candidate.get('model') == MODEL_ID:
                payload = candidate
        except Exception:
            payload = None
    if payload is None:
        payload = {'video_id': video_id, 'source': str(video_dir), 'model': MODEL_ID,
                   'language': 'vi', 'min_confidence': MIN_CONFIDENCE, 'keep_floor': KEEP_FLOOR,
                   'has_keyframe_map': mapping is not None, 'complete': False,
                   'keyframe_count': 0, 'keyframes': []}

    done = {item['keyframe'] for item in payload['keyframes']}
    todo = [p for p in images if p.name not in done]
    processed = 0
    for image_path, image in iter_loaded(todo, prefetch=CFG['io_workers'] * 3):
        detections = ocr_image(image) if image is not None else []
        order = keyframe_order(image_path)
        mapped = mapping.get(order) if mapping and order is not None else None
        payload['keyframes'].append({
            'keyframe': image_path.name, 'n': order,
            'frame_idx': mapped['frame_idx'] if mapped else None,
            'pts_time': mapped['pts_time'] if mapped else None,
            'fps': mapped['fps'] if mapped else None,
            'text': ' '.join(d['text'] for d in detections if d['kept']),
            'detections': detections})
        processed += 1
        # Ghi theo lô: /kaggle/working là SSD local nên rẻ, nhưng ghi lại cả payload sau MỖI
        # ảnh vẫn tốn (payload lớn dần) — đó chính là chỗ bản Colab bị chậm.
        if processed % FLUSH_EVERY == 0:
            payload['keyframe_count'] = len(payload['keyframes'])
            atomic_write(partial, payload)

    # Giữ đúng thứ tự keyframe kể cả khi resume chen giữa chừng.
    order_of = {p.name: i for i, p in enumerate(images)}
    payload['keyframes'].sort(key=lambda k: order_of.get(k['keyframe'], 1 << 30))
    payload['keyframe_count'] = len(payload['keyframes'])
    payload['complete'] = True
    atomic_write(output_json_path(video_id), payload)
    if partial.exists():
        partial.unlink()
    return payload, processed

# ---------------------------------------------------------------- preview
if args.preview > 0:
    from PIL import ImageDraw, ImageFont
    from matplotlib import font_manager

    preview_dir = Path(CFG['preview_dir'])
    preview_dir.mkdir(parents=True, exist_ok=True)
    font_path = font_manager.findfont('DejaVu Sans')   # font mặc định của PIL không có dấu tiếng Việt

    # Lấy mẫu rải trên nhiều video; một video thể thao có thể gần như không có chữ Việt có dấu.
    candidates = []
    for candidate_video in my_videos[:min(20, len(my_videos))]:
        images = list_images(candidate_video)
        step = max(1, len(images) // max(1, args.preview))
        candidates.extend((candidate_video, path) for path in images[::step][:args.preview])

    scanned = []
    for candidate_video, path in candidates:
        image = load_image(path)
        if image is None:
            continue
        detections = ocr_image(image)
        kept = [d for d in detections if d['kept']]
        if kept:
            accent_count = sum(
                len(unicodedata.normalize('NFD', character)) > 1
                for detection in kept for character in detection['text'] if character.isalpha()
            )
            scanned.append((candidate_video, path, image, detections, accent_count))
    scanned.sort(key=lambda item: (-item[4], -sum(d['kept'] for d in item[3])))

    preview_rows = scanned[:args.preview]
    for video_dir, path, image, detections, _ in preview_rows:
        canvas = image.copy()
        draw = ImageDraw.Draw(canvas)
        font = ImageFont.truetype(font_path, max(13, canvas.height // 45))
        line_width = max(2, canvas.height // 400)
        for d in detections:
            x1, y1, x2, y2 = d['box']
            color = (0, 255, 0) if d['kept'] else (255, 60, 60)
            draw.rectangle([x1, y1, x2, y2], outline=color,
                           width=line_width if d['kept'] else max(1, line_width - 1))
            label = f"{d['text']} ({d['confidence']:.2f})"
            left, top, right, bottom = draw.textbbox((0, 0), label, font=font)
            label_w, label_h = right - left, bottom - top
            label_y = y1 - label_h - 3
            if label_y < 0:
                label_y = min(y2 + 2, canvas.height - label_h - 1)
            label_x = min(x1, max(0, canvas.width - label_w - 2))
            draw.rectangle([label_x, label_y, label_x + label_w + 3, label_y + label_h + 3], fill=color)
            draw.text((label_x + 2, label_y + 1), label, fill=(0, 0, 0), font=font)
        saved = preview_dir / f'{video_dir.name}_{path.stem}.jpg'
        canvas.save(saved, quality=92)
        log('preview', saved, '|', ' '.join(d['text'] for d in detections if d['kept'])[:120])
    if not scanned:
        log('Không tìm thấy keyframe nào có chữ trong tập preview')
    else:
        letters = [character for _, _, _, detections, _ in preview_rows
                   for detection in detections if detection['kept']
                   for character in detection['text'] if character.isalpha()]
        accented = sum(len(unicodedata.normalize('NFD', character)) > 1 for character in letters)
        ratio = 100 * accented / max(1, len(letters))
        log(f'PREVIEW: {accented}/{len(letters)} ký tự chữ cái có dấu ({ratio:.1f}%). '
            'Phải kiểm tra ảnh bằng mắt trước khi chạy full.')
    sys.exit(0)

# ---------------------------------------------------------------- vòng chính
pending = []
for video_dir in my_videos:
    destination = output_json_path(video_dir.name)
    if destination.exists():
        try:
            existing = json.loads(destination.read_text(encoding='utf-8'))
        except Exception:
            existing = {}
        if existing.get('model') == MODEL_ID and existing.get('complete'):
            continue
    pending.append(video_dir)

if args.limit_videos > 0:
    pending = pending[:args.limit_videos]

log(f'{len(my_videos) - len(pending)} video đã xong, còn {len(pending)}')

images_done = ocr_seconds = 0.0
videos_done = failed = 0
failures = []
stopped_early = False

for index, video_dir in enumerate(pending, 1):
    if DEADLINE and time.time() > DEADLINE:
        log(f'Chạm mốc {CFG["max_run_hours"]}h -> dừng để mục 6-7 kịp chạy. '
            f'Còn {len(pending) - index + 1} video.')
        stopped_early = True
        break
    started = time.time()
    try:
        payload, processed = ocr_video(video_dir)
        elapsed = time.time() - started
        ocr_seconds += elapsed
        images_done += processed
        videos_done += 1
        with_text = sum(bool(k['text']) for k in payload['keyframes'])
        rate = processed / elapsed if elapsed > 0 else 0
        log(f'[{index}/{len(pending)}] {video_dir.name}: {with_text}/{payload["keyframe_count"]} '
            f'ảnh có chữ | {processed} ảnh trong {elapsed:.0f}s ({rate:.1f} img/s)')
    except Exception as exc:
        import traceback
        failed += 1
        failures.append({'video_id': video_dir.name, 'error': repr(exc)})
        log(f'[{index}/{len(pending)}] LỖI {video_dir.name}: {exc!r}')
        traceback.print_exc()
    finally:
        if use_gpu:
            torch.cuda.empty_cache()

if failures:
    atomic_write(OUTPUT_ROOT / f'_failed_shard{args.shard_index}.json', failures)

overall = images_done / ocr_seconds if ocr_seconds > 0 else 0
log(f'SHARD DONE videos={videos_done} failed={failed} images={int(images_done)} '
    f'seconds={ocr_seconds:.0f} rate={overall:.2f} stopped_early={int(stopped_early)}')
sys.exit(0)

In [ ]:
# Gói cấu hình thành file cho worker đọc — dễ kiểm tra và tránh dòng lệnh dài loằng ngoằng.
WORKER_PATH = '/kaggle/working/ocr_worker.py'
CONFIG_PATH = '/kaggle/working/_worker_config.json'

worker_config = {
    'output_root': str(OUTPUT_ROOT),
    'map_dir': str(MAP_KEYFRAMES_DIRECTORY),
    'keyframe_roots': [str(p) for p in KEYFRAME_ROOTS],
    'model_dir': str(MODEL_DIR),
    'vietocr_weights': str(VIETOCR_WEIGHTS),
    'preview_dir': str(PREVIEW_DIR),
    'model_id': MODEL_ID,
    'min_confidence': MIN_CONFIDENCE,
    'keep_floor': KEEP_FLOOR,
    'box_padding': BOX_PADDING,
    'box_padding_x_ratio': BOX_PADDING_X_RATIO,
    'box_padding_y_ratio': BOX_PADDING_Y_RATIO,
    'min_box_side': MIN_BOX_SIDE,
    'recognition_batch': RECOGNITION_BATCH,
    'io_workers': IO_WORKERS,
    'flush_every': FLUSH_EVERY,
    'max_run_hours': MAX_RUN_HOURS,
    'image_extensions': sorted(IMAGE_EXTENSIONS),
    'detect': {'min_size': MIN_SIZE, 'text_threshold': TEXT_THRESHOLD, 'low_text': LOW_TEXT,
               'link_threshold': LINK_THRESHOLD, 'canvas_size': CANVAS_SIZE, 'mag_ratio': MAG_RATIO},
}
Path(CONFIG_PATH).write_text(json.dumps(worker_config, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(worker_config, ensure_ascii=False, indent=2))

## 6A. Test tốc độ trên 1 video (tùy chọn)

Đặt RUN_SPEED_TEST = True ở cell kế tiếp nếu muốn đo ETA. Preview chất lượng tại mục 6B là bắt buộc.


In [ ]:
import re as _re
import subprocess
import sys
import time

def run_worker(extra_args, shard_index=0, shard_count=1, gpu=0, stream=True):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    command = [sys.executable, WORKER_PATH, '--config', CONFIG_PATH,
               '--shard-index', str(shard_index), '--shard-count', str(shard_count)] + extra_args
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if stream:
            print(line, end='', flush=True)
    proc.wait()
    return proc.returncode, ''.join(lines)

# Test này ghi kết quả một video vào output; lần chạy full sẽ tự bỏ qua video đã xong.
RUN_SPEED_TEST = False

if RUN_SPEED_TEST:
    started = time.time()
    code_, output = run_worker(['--limit-videos', '1'])
    print(f'\nWorker thoát mã {code_} sau {time.time() - started:.0f}s')
    assert code_ == 0, 'Speed test thất bại; không chạy full.'
    match = _re.search(r'rate=([\d.]+)', output)
    if match and float(match.group(1)) > 0:
        rate = float(match.group(1))
        finished = {p.stem for p in OUTPUT_ROOT.glob('L*_V*.json')
                    if not p.name.endswith('.partial.json')}
        remaining = sum(IMAGE_COUNTS[v] for v in video_dirs if v.name not in finished)
        hours = remaining / (rate * NUM_WORKERS) / 3600
        print(f'Tốc độ: {rate:.2f} ảnh/giây/GPU | ETA: {hours:.1f}h với {NUM_WORKERS} GPU')
    else:
        print('Không đọc được tốc độ; kiểm tra log phía trên.')
else:
    print('Speed test đang tắt. Đặt RUN_SPEED_TEST = True nếu muốn đo ETA.')

In [ ]:
# 6B — PREVIEW CHẤT LƯỢNG BẮT BUỘC
# XANH = giữ trong OCR index | ĐỎ = dưới MIN_CONFIDENCE.
# Không chạy toàn dataset và không ghi JSON kết quả; chỉ tạo ảnh preview tạm.
from IPython.display import Image as IPyImage, display

PREVIEW_COUNT = 8

# Tránh hiển thị nhầm preview của cấu hình cũ.
for old_preview in PREVIEW_DIR.glob('*.jpg'):
    old_preview.unlink()

preview_started = time.time()
preview_code, preview_log = run_worker(['--preview', str(PREVIEW_COUNT)], stream=True)
assert preview_code == 0, 'Preview worker thất bại; xem log phía trên.'

previews = sorted(PREVIEW_DIR.glob('*.jpg'))
assert previews, 'Không tạo được preview. Kiểm tra detector, model và dataset.'

print(f'\n=== {len(previews)} PREVIEW — kiểm tra box và dấu tiếng Việt bằng mắt ===')
for path in previews:
    print(path.name)
    display(IPyImage(filename=str(path), width=1100))

print(f'Preview hoàn tất sau {time.time() - preview_started:.0f}s.')
print('Chữ nhỏ không có box: tăng MAG_RATIO hoặc hạ LOW_TEXT nhẹ.')
print('Box đúng nhưng sai dấu: không chạy full; kiểm tra crop/padding và weight VietOCR.')
print('Nếu kết quả đạt yêu cầu, chạy cell xác nhận ngay bên dưới.')

## 6C. Xác nhận preview trước khi chạy toàn bộ

Chỉ đổi thành True sau khi box phủ trọn chữ, dấu tiếng Việt đúng và mức nhiễu chấp nhận được.


In [ ]:
# Đổi False -> True sau khi đã xem trực tiếp toàn bộ ảnh preview ở mục 6B.
PREVIEW_APPROVED = False

assert PREVIEW_APPROVED, (
    'Chưa xác nhận chất lượng. Xem preview ở mục 6B rồi đổi PREVIEW_APPROVED = True.'
)
print('✓ Preview đã được xác nhận. Có thể chạy mục 7.')

## 7. Chạy toàn bộ trên cả 2 GPU

Mỗi worker nhận một shard round-robin và chỉ ghi file của video mình phụ trách → hai process
không bao giờ ghi đè nhau, không cần khoá.

Cell này chạy nhiều giờ. Log của hai worker xen kẽ nhau, phân biệt bằng tiền tố `[w0]` / `[w1]`.

In [ ]:
assert globals().get('PREVIEW_APPROVED', False), (
    'BỊ CHẶN: chạy mục 6B, kiểm tra ảnh, rồi đặt PREVIEW_APPROVED = True ở mục 6C.'
)
import threading

def launch_all():
    procs, lock = [], threading.Lock()
    for shard in range(NUM_WORKERS):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(shard % max(1, torch.cuda.device_count())))
        proc = subprocess.Popen(
            [sys.executable, WORKER_PATH, '--config', CONFIG_PATH,
             '--shard-index', str(shard), '--shard-count', str(NUM_WORKERS)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        procs.append(proc)

    def pump(proc):
        for line in proc.stdout:
            with lock:                    # tránh hai worker cắt ngang giữa dòng của nhau
                print(line, end='', flush=True)

    threads = [threading.Thread(target=pump, args=(p,), daemon=True) for p in procs]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return [p.wait() for p in procs]

run_started = time.time()
codes = launch_all()
print(f'\n=== Tất cả worker đã thoát: {codes} sau {(time.time() - run_started) / 3600:.2f}h')
if any(c != 0 for c in codes):
    print('CÓ WORKER LỖI — vẫn chạy tiếp mục 8-9 để giữ phần đã xong, rồi xem log ở trên.')

## 8. Ghép thành `ocr_index.jsonl`

File này mới là thứ bước index/search dùng. Các `L*_V*.json` là dữ liệu thô (giữ cả box dưới
ngưỡng), nặng hơn nhiều.

Muốn đổi `MIN_CONFIDENCE` về sau: sửa `THRESHOLD` ngay dưới rồi chạy lại **chỉ** cell này —
không phải đụng đến GPU, vì JSON đã giữ sẵn mọi box từ `KEEP_FLOOR` trở lên.

In [ ]:
THRESHOLD = MIN_CONFIDENCE     # đổi số này rồi chạy lại cell để lọc lại mà không cần GPU

jsonl_path = Path('/kaggle/working/ocr_index.jsonl')
rows = no_frame_idx = incomplete = 0
video_ids = set()

with jsonl_path.open('w', encoding='utf-8') as handle:
    for json_path in sorted(OUTPUT_ROOT.glob('L*_V*.json')):
        if json_path.name.endswith('.partial.json'):
            continue
        payload = json.loads(json_path.read_text(encoding='utf-8'))
        if payload.get('model') != MODEL_ID or not payload.get('complete'):
            incomplete += 1
            continue
        video_ids.add(payload['video_id'])
        for keyframe in payload['keyframes']:
            text = ' '.join(d['text'] for d in keyframe['detections']
                            if d['confidence'] >= THRESHOLD)
            if not text:
                continue
            no_frame_idx += keyframe['frame_idx'] is None
            handle.write(json.dumps({'video_id': payload['video_id'],
                                     'frame_idx': keyframe['frame_idx'],
                                     'pts_time': keyframe['pts_time'],
                                     'keyframe': keyframe['keyframe'],
                                     'text': text}, ensure_ascii=False) + '\n')
            rows += 1

print(f'{rows} dòng từ {len(video_ids)} video -> {jsonl_path} '
      f'({jsonl_path.stat().st_size / 1e6:.1f} MB)')
if incomplete:
    print(f'Bỏ qua {incomplete} file chưa complete / khác model')
if no_frame_idx:
    print(f'CẢNH BÁO: {no_frame_idx} dòng thiếu frame_idx (video không có map-keyframes)')

leftover = sorted(p.name for p in OUTPUT_ROOT.glob('*.partial.json'))
if leftover:
    print(f'\n{len(leftover)} video còn DANG DỞ, KHÔNG có trong index:', ', '.join(leftover[:10]))
missing_videos = sorted(v.name for v in video_dirs if v.name not in video_ids)
if missing_videos:
    print(f'{len(missing_videos)} video chưa OCR:', ', '.join(missing_videos[:10]),
          '...' if len(missing_videos) > 10 else '')

## 9. Đóng gói `.zip`

Một file zip để tải về, thay vì hàng nghìn JSON lẻ.

In [ ]:
import zipfile

ARCHIVE = Path('/kaggle/working/ocr_easyocr_vietocr.zip')
TMP_ARCHIVE = SCRATCH / 'ocr_easyocr_vietocr.zip'   # nén ở scratch rồi mới move sang working

files = [p for p in sorted(OUTPUT_ROOT.glob('*.json')) if not p.name.endswith('.partial.json')]
assert files, 'Chưa có kết quả nào để đóng gói'

with zipfile.ZipFile(TMP_ARCHIVE, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in files:
        zf.write(path, arcname=f'OCR_EasyOCR_VietOCR/{path.name}')
    zf.write(jsonl_path, arcname='ocr_index.jsonl')

shutil.move(str(TMP_ARCHIVE), str(ARCHIVE))
raw_mb = sum(p.stat().st_size for p in files) / 1e6
print(f'{len(files)} file JSON ({raw_mb:.0f} MB thô) + ocr_index.jsonl')
print(f'-> {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('\nTải về: panel Output bên phải -> ocr_easyocr_vietocr.zip -> nút Download')

## 10. Chạy tiếp ở session sau (nếu 12h không đủ)

1. **Save Version** → *Save & Run All* hoặc *Quick Save*. Chỉ khi lưu được thì `/kaggle/working`
   mới thành output bền vững; session interactive hết hạn là mất sạch.
2. Ở notebook session sau: *Add Input* → chọn output của version vừa lưu.
3. Điền vào mục 1:

   ```python
   RESUME_FROM = ['/kaggle/input/<tên-version-vừa-add>/OCR_EasyOCR_VietOCR']
   ```

4. Chạy lại từ đầu. Mục 4 chép các video đã xong vào, worker tự bỏ qua chúng và chỉ làm phần còn lại.

**Nếu ETA ở mục 6 cho thấy quá lâu**, chia đợt theo khối lượng thay vì theo tên — `L26` chiếm
~45% và `L25` ~21% toàn dataset, nên cắt kiểu "L21-L25 / L26-L30" sẽ lệch 1:3:

```python
TARGET_FOLDERS = ['Keyframes_L26_a', 'Keyframes_L26_b', 'Keyframes_L26_c',
                  'Keyframes_L26_d', 'Keyframes_L26_e',
                  'Keyframes_L27', 'Keyframes_L23']          # đợt 1 (~49%)

TARGET_FOLDERS = ['Keyframes_L25', 'Keyframes_L21', 'Keyframes_L22',
                  'Keyframes_L24', 'Keyframes_L28', 'Keyframes_L29',
                  'Keyframes_L30']                            # đợt 2 (~51%)
```